# Experiment 1: Concentration and Repeatability Testing Among Sensors

## Overview
Assess sensor response to varying Salmonella concentrations (0, 10, 100, 1000 CFU/ml) and evaluate signal repeatability across multiple sensors. This experiment tests 15 sensors per concentration across 3 serovars (Typhimurium, Hadar, Muenchen) in turkey rinsate.

**Protocol Alignment**: This notebook implements Experiment 1 from the Fiber Optics SERS Sensor Testing Protocol, focusing on concentration-response relationships and cross-sensor repeatability.

## Key Concepts
- **Concentration-response relationship**: How signal intensity changes with Salmonella concentration
- **Cross-sensor repeatability**: Consistency of measurements across different sensors at the same concentration
- **Detection analysis**: Binary classification (positive/negative) and detection metrics
- **Serovar comparison**: Differences in signal response across serovars

## Objectives (Protocol Requirements)
1. ✅ Load and organize SERS data for multiple serovars (ST, Hadar, Muenchen) at target concentrations (0, 10, 100, 1000 CFU/ml)
2. ✅ **Detection analysis**: Determine detection status (positive/negative) and calculate detection metrics
3. ✅ **Concentration-response analysis**: Plot signal intensity vs concentration, estimate detection limits
4. ✅ **Sensor repeatability**: Evaluate signal repeatability across multiple sensors at each concentration
5. ✅ **Serovar comparison**: Compare signal responses across serovars
6. ✅ **Summary statistics**: Per-concentration, per-serotype, and per-sensor summaries

## Protocol Compliance
This notebook implements all requirements for Experiment 1:
- ✅ Detection status determination (positive/negative)
- ✅ Concentration estimation and calibration
- ✅ Signal repeatability across sensors
- ✅ Training data generation for ML modeling

## Configuration
- **Target Concentrations**: 0, 10, 100, 1000 CFU/ml (as per Protocol)
- **Serovars**: Typhimurium (ST), Hadar, Muenchen
- **Sample Type**: Turkey rinsate from Cargill
- **Testing Strategy**: 15 sensors × 4 concentrations × 3 serovars × 2 samples = 360 tests

Notes:
- This notebook is fully standalone and self-contained.
- All functions are defined within this notebook to ensure reproducibility.


In [ ]:
# Imports and Setup
# --------------------------------------------------------------------------------------------------
# All imports required for this notebook (fully standalone)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import integrate
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
import warnings

# Project-specific imports
from sensd_sers_analysis.data import load_dataset_by_serotypes
from sensd_sers_analysis.utils import natural_sort

warnings.filterwarnings("ignore")

# Plotting style
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ All imports completed successfully")
print(
    "📦 Required packages: numpy, pandas, matplotlib, seaborn, scipy, scikit-learn, sensd_sers_analysis"
)

## 1. Data Loading and Inspection

Load SERS data for all three serovars (Typhimurium/ST, Hadar, Muenchen) at target concentrations (0, 10, 100, 1000 CFU/ml). The Protocol specifies these exact concentrations, but we'll analyze all available concentrations for robustness.


In [ ]:
# Load SERS Data (Experiment 1 focus)
# --------------------------------------------------------------------------------------------------

# Configuration: Protocol specifies 0, 10, 100, 1000 CFU/ml
# Set to None to analyze all concentrations, or specify list to filter
TARGET_CONCENTRATIONS = (
    None  # Protocol: [0, 10, 100, 1000] (set to None to include all concentrations)
)
TOLERANCE = 0.1  # Tolerance for concentration matching

# Define dataset location
data_folder = "../example_data/SERS Data 8 (April 2025)/"
signals_folders = ["March testing-Dilutions"]  # Adjust based on available data

# Protocol requires: ST (Typhimurium), Hadar, Muenchen
# Note: Check actual serotype labels in metadata - they might be "ST", "Hadar", "Muenchen" or different
serotypes = ["ST"]  # Start with ST, will expand after checking available serotypes

try:
    # First, check what serotypes are available
    import pandas as pd
    from pathlib import Path

    signals_path = Path(data_folder) / signals_folders[0]
    metadata_file = signals_path / "_metadata.xlsx"
    if metadata_file.exists():
        meta = pd.read_excel(metadata_file)
        available_serotypes = sorted(meta["Serotype"].unique().tolist())
        print(f"📊 Available serotypes in metadata: {available_serotypes}")
        print("   Protocol requires: ST (Typhimurium), Hadar, Muenchen")
        print("   Note: Serotype labels may differ - adjust 'serotypes' list accordingly")

    # Load data
    data_df = load_dataset_by_serotypes(data_folder, signals_folders, serotypes)
    print(f"\n✅ Successfully loaded {len(data_df)} data entries")

    # Optional concentration filtering
    if TARGET_CONCENTRATIONS is not None:
        original_count = len(data_df)
        mask = np.zeros(len(data_df), dtype=bool)
        for target_conc in TARGET_CONCENTRATIONS:
            mask |= np.abs(data_df["concentration"] - target_conc) <= TOLERANCE
        data_df = data_df[mask].copy()
        print(
            f"📊 Filtered to target concentrations {TARGET_CONCENTRATIONS} CFU/ml: {len(data_df)} entries (from {original_count})"
        )

except Exception as e:
    print(f"❌ Error loading data: {e}")
    import traceback

    traceback.print_exc()
    data_df = pd.DataFrame()

# Basic inspection
if not data_df.empty:
    print("\n📊 Dataset Structure Analysis:")
    print("=" * 60)
    print(f"Number of data entries: {len(data_df)}")
    print(f"Raman shift points: {len(data_df['raman_shift'].iloc[0])}")
    print(
        f"Raman shift range: {data_df['raman_shift'].iloc[0][0]:.1f} - "
        f"{data_df['raman_shift'].iloc[0][-1]:.1f} cm⁻¹"
    )
    print(f"Columns: {list(data_df.columns)}")

    # Show concentration distribution
    conc_counts = data_df["concentration"].value_counts().sort_index()
    print("\nConcentration distribution:")
    for conc, count in conc_counts.head(20).items():
        print(f"  {conc:.1f} CFU/mL: {count} entries")
    if len(conc_counts) > 20:
        print(f"  ... and {len(conc_counts) - 20} more concentrations")

    # Show serotype distribution
    serotype_counts = data_df["serotype"].value_counts()
    print("\nSerotype distribution:")
    for sero, count in serotype_counts.items():
        print(f"  {sero}: {count} entries")

    # Show sensor distribution
    sensor_ids = natural_sort([str(s) for s in data_df["sensor_id"].unique().tolist()])
    print(
        f"\nSensors ({len(sensor_ids)} total): {sensor_ids[:15]}{'...' if len(sensor_ids) > 15 else ''}"
    )

    # Protocol check
    print("\n📋 Protocol Requirements Check:")
    print("  Required concentrations: [0, 10, 100, 1000] CFU/ml")
    required_concs = [0, 10, 100, 1000]
    available_concs = set(data_df["concentration"].unique())
    missing_concs = [
        c
        for c in required_concs
        if not any(abs(avail - c) <= TOLERANCE for avail in available_concs)
    ]
    if missing_concs:
        print(f"  ⚠️ Missing concentrations: {missing_concs}")
    else:
        print("  ✅ All required concentrations present")

    print("  Required serovars: ST (Typhimurium), Hadar, Muenchen")
    print(f"  Available serovars: {list(serotype_counts.index)}")

else:
    print("❌ No data loaded - cannot proceed")

## 2. Signal Feature Extraction

Extract features from SERS spectra for analysis. Features include:
- Signal intensity metrics (mean, max, min, std, range)
- Area under curve (AUC)
- Peak characteristics
- Signal-to-noise ratio (SNR)


In [ ]:
# Signal Feature Extraction
# --------------------------------------------------------------------------------------------------


def extract_signal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Extract features from SERS signals for Experiment 1 analysis.

    Features extracted:
    - Signal intensity: mean, max, min, std, range
    - Area under curve (AUC)
    - Peak characteristics
    - Signal-to-noise ratio (SNR)

    Returns DataFrame with original columns plus feature columns.
    """
    if df.empty:
        return df

    features_df = df.copy()

    # Extract features for each signal
    signal_means = []
    signal_maxs = []
    signal_mins = []
    signal_stds = []
    signal_ranges = []
    signal_aucs = []
    signal_snrs = []

    for idx, row in df.iterrows():
        signal = np.asarray(row["signal"], dtype=float)
        raman_shift = np.asarray(row["raman_shift"], dtype=float)

        # Basic statistics
        signal_mean = float(np.mean(signal))
        signal_max = float(np.max(signal))
        signal_min = float(np.min(signal))
        signal_std = float(np.std(signal))
        signal_range = signal_max - signal_min

        # Area under curve (trapezoidal integration)
        signal_auc = float(integrate.trapezoid(signal, raman_shift))

        # Signal-to-noise ratio (using std as noise estimate)
        signal_snr = signal_mean / signal_std if signal_std > 0 else 0.0

        signal_means.append(signal_mean)
        signal_maxs.append(signal_max)
        signal_mins.append(signal_min)
        signal_stds.append(signal_std)
        signal_ranges.append(signal_range)
        signal_aucs.append(signal_auc)
        signal_snrs.append(signal_snr)

    # Add feature columns
    features_df["signal_mean"] = signal_means
    features_df["signal_max"] = signal_maxs
    features_df["signal_min"] = signal_mins
    features_df["signal_std"] = signal_stds
    features_df["signal_range"] = signal_ranges
    features_df["signal_auc"] = signal_aucs
    features_df["signal_snr"] = signal_snrs

    return features_df


# Extract features
if not data_df.empty:
    data_df = extract_signal_features(data_df)
    print(f"✅ Signal features extracted for {len(data_df)} entries")
    print(
        "   Features: signal_mean, signal_max, signal_min, signal_std, signal_range, signal_auc, signal_snr"
    )
else:
    print("⚠️ No data available for feature extraction")

## 3. Detection Analysis (Protocol Requirement)

**Protocol Objective**: Determine detection status (positive/negative)

For each measurement, determine if Salmonella was detected (positive) or not (negative). A measurement is considered positive if concentration > 0 CFU/ml.


In [ ]:
# Detection Analysis
# --------------------------------------------------------------------------------------------------


def determine_detection_status(df: pd.DataFrame, threshold: float = 0.0) -> pd.DataFrame:
    """Determine detection status (positive/negative) for each measurement.

    Args:
        df: DataFrame with concentration column
        threshold: Concentration threshold for positive detection (default: 0.0)

    Returns:
        DataFrame with added 'detection_status' column (1 = positive, 0 = negative)
    """
    df = df.copy()
    df["detection_status"] = (df["concentration"] > threshold).astype(int)
    df["detection_label"] = df["detection_status"].map({1: "Positive", 0: "Negative"})
    return df


def calculate_detection_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Calculate detection performance metrics.

    Args:
        y_true: True labels (1 = positive, 0 = negative)
        y_pred: Predicted labels (1 = positive, 0 = negative)

    Returns:
        Dictionary with detection metrics
    """
    if len(y_true) == 0 or len(y_pred) == 0:
        return {}

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # Detection-specific metrics
    sensitivity = recall  # Same as recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    # Detection index (Youden's J statistic)
    detection_index = sensitivity + specificity - 1

    return {
        "true_positives": int(tp),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1_score": float(f1),
        "false_positive_rate": float(false_positive_rate),
        "false_negative_rate": float(false_negative_rate),
        "detection_index": float(detection_index),
    }


def print_detection_report(metrics: dict, title: str = "Detection Report") -> None:
    """Print formatted detection report."""
    print("\n" + "=" * 70)
    print(f"DETECTION REPORT - {title}")
    print("=" * 70)
    print("\nConfusion Matrix:")
    print(f"  True Negatives:  {metrics['true_negatives']}")
    print(f"  False Positives: {metrics['false_positives']}")
    print(f"  False Negatives: {metrics['false_negatives']}")
    print(f"  True Positives:  {metrics['true_positives']}")

    print("\nPerformance Metrics:")
    print(f"  Accuracy:         {metrics['accuracy']:.4f}")
    print(f"  Precision:        {metrics['precision']:.4f}")
    print(f"  Recall (Sensitivity): {metrics['recall']:.4f}")
    print(f"  Specificity:     {metrics['specificity']:.4f}")
    print(f"  F1-Score:        {metrics['f1_score']:.4f}")

    print("\nDetection Indices:")
    print(f"  Detection Index (Youden's J): {metrics['detection_index']:.4f}")
    print(f"  False Positive Rate: {metrics['false_positive_rate']:.4f}")
    print(f"  False Negative Rate: {metrics['false_negative_rate']:.4f}")


# Determine detection status
if not data_df.empty:
    data_df = determine_detection_status(data_df, threshold=0.0)

    # Overall detection metrics
    y_true = data_df["detection_status"].values
    y_pred = data_df["detection_status"].values  # For now, using ground truth (concentration > 0)

    overall_metrics = calculate_detection_metrics(y_true, y_pred)
    print_detection_report(overall_metrics, "Experiment 1 - Overall Detection")

    # Detection by concentration
    print("\n" + "=" * 70)
    print("DETECTION BY CONCENTRATION")
    print("=" * 70)
    for conc in sorted(data_df["concentration"].unique()):
        conc_data = data_df[data_df["concentration"] == conc]
        n_total = len(conc_data)
        n_positive = conc_data["detection_status"].sum()
        detection_rate = n_positive / n_total if n_total > 0 else 0.0
        print(f"\nConcentration: {conc:.1f} CFU/ml")
        print(f"  Total samples: {n_total}")
        print(f"  Positive detections: {n_positive} ({detection_rate * 100:.1f}%)")
        print(f"  Negative detections: {n_total - n_positive} ({(1 - detection_rate) * 100:.1f}%)")

    # Detection by serotype
    print("\n" + "=" * 70)
    print("DETECTION BY SEROTYPE")
    print("=" * 70)
    for sero in sorted(data_df["serotype"].unique()):
        sero_data = data_df[data_df["serotype"] == sero]
        n_total = len(sero_data)
        n_positive = sero_data["detection_status"].sum()
        detection_rate = n_positive / n_total if n_total > 0 else 0.0
        print(f"\nSerotype: {sero}")
        print(f"  Total samples: {n_total}")
        print(f"  Positive detections: {n_positive} ({detection_rate * 100:.1f}%)")
else:
    print("⚠️ No data available for detection analysis")

## 4. Concentration-Response Analysis (Protocol Requirement)

**Protocol Objective**: Estimate Salmonella concentrations

Plot signal intensity vs concentration to establish concentration-response relationships and estimate detection limits.


In [ ]:
# Concentration-Response Analysis
# --------------------------------------------------------------------------------------------------


def plot_concentration_response(df: pd.DataFrame, feature: str = "signal_mean") -> None:
    """Plot concentration-response relationship.

    Args:
        df: DataFrame with concentration and feature columns
        feature: Feature to plot (default: "signal_mean")
    """
    if df.empty or feature not in df.columns:
        print(f"⚠️ Cannot plot: feature '{feature}' not available")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        f"Experiment 1: Concentration-Response Analysis ({feature.replace('_', ' ').title()})",
        fontsize=16,
        fontweight="bold",
    )

    # Plot 1: Scatter plot (all data points)
    ax1 = axes[0, 0]
    for serotype in df["serotype"].unique():
        sero_data = df[df["serotype"] == serotype]
        ax1.scatter(sero_data["concentration"], sero_data[feature], alpha=0.5, label=serotype, s=30)
    ax1.set_xlabel("Concentration (CFU/ml)")
    ax1.set_ylabel(feature.replace("_", " ").title())
    ax1.set_title("Concentration-Response: All Data Points")
    ax1.set_xscale("log")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Mean ± std per concentration
    ax2 = axes[0, 1]
    concentrations = sorted(df["concentration"].unique())
    for serotype in df["serotype"].unique():
        sero_data = df[df["serotype"] == serotype]
        means = []
        stds = []
        concs = []
        for conc in concentrations:
            conc_data = sero_data[sero_data["concentration"] == conc]
            if len(conc_data) > 0:
                means.append(conc_data[feature].mean())
                stds.append(conc_data[feature].std())
                concs.append(conc)
        if concs:
            ax2.errorbar(
                concs,
                means,
                yerr=stds,
                marker="o",
                label=serotype,
                capsize=5,
                linewidth=2,
                markersize=8,
            )
    ax2.set_xlabel("Concentration (CFU/ml)")
    ax2.set_ylabel(feature.replace("_", " ").title())
    ax2.set_title("Concentration-Response: Mean ± Std")
    ax2.set_xscale("log")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Plot 3: Boxplot per concentration
    ax3 = axes[1, 0]
    # Group by concentration bins for better visualization
    conc_bins = []
    bin_labels = []
    for conc in concentrations:
        conc_data = df[df["concentration"] == conc]
        if len(conc_data) > 0:
            conc_bins.append(conc_data[feature].values)
            bin_labels.append(f"{conc:.0f}")

    if conc_bins:
        bp = ax3.boxplot(conc_bins, labels=bin_labels, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("lightblue")
            patch.set_alpha(0.7)
        ax3.set_xlabel("Concentration (CFU/ml)")
        ax3.set_ylabel(feature.replace("_", " ").title())
        ax3.set_title("Concentration-Response: Distribution per Concentration")
        ax3.grid(True, axis="y", alpha=0.3)

    # Plot 4: Per-serotype comparison
    ax4 = axes[1, 1]
    for serotype in df["serotype"].unique():
        sero_data = df[df["serotype"] == serotype]
        concs = sorted(sero_data["concentration"].unique())
        means = [sero_data[sero_data["concentration"] == c][feature].mean() for c in concs]
        ax4.plot(concs, means, marker="o", label=serotype, linewidth=2, markersize=8)
    ax4.set_xlabel("Concentration (CFU/ml)")
    ax4.set_ylabel(feature.replace("_", " ").title())
    ax4.set_title("Concentration-Response: Per-Serotype Comparison")
    ax4.set_xscale("log")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def estimate_detection_limits(df: pd.DataFrame, feature: str = "signal_mean") -> dict:
    """Estimate detection limits (LOD, LOQ) based on signal variability.

    LOD (Limit of Detection): 3× standard deviation of blank/negative control
    LOQ (Limit of Quantification): 10× standard deviation of blank/negative control

    Args:
        df: DataFrame with concentration and feature columns
        feature: Feature to use for detection limit estimation

    Returns:
        Dictionary with LOD and LOQ estimates
    """
    if df.empty or feature not in df.columns:
        return {}

    # Get negative control (concentration = 0 or very low)
    negative_control = df[df["concentration"] <= 0.1]

    if len(negative_control) == 0:
        print("⚠️ No negative control data available for LOD/LOQ estimation")
        return {}

    # Calculate baseline and noise
    baseline_mean = negative_control[feature].mean()
    baseline_std = negative_control[feature].std()

    # LOD = baseline + 3×std
    lod = baseline_mean + 3 * baseline_std

    # LOQ = baseline + 10×std
    loq = baseline_mean + 10 * baseline_std

    # Find concentrations that correspond to LOD and LOQ
    # (approximate by finding where signal crosses these thresholds)
    positive_data = df[df["concentration"] > 0.1]
    if len(positive_data) > 0:
        # Find concentration where signal first exceeds LOD
        sorted_data = positive_data.sort_values("concentration")
        lod_conc = None
        loq_conc = None

        for _, row in sorted_data.iterrows():
            if lod_conc is None and row[feature] >= lod:
                lod_conc = row["concentration"]
            if loq_conc is None and row[feature] >= loq:
                loq_conc = row["concentration"]
            if lod_conc is not None and loq_conc is not None:
                break
    else:
        lod_conc = None
        loq_conc = None

    return {
        "baseline_mean": baseline_mean,
        "baseline_std": baseline_std,
        "lod_signal": lod,
        "loq_signal": loq,
        "lod_concentration": lod_conc,
        "loq_concentration": loq_conc,
        "n_negative_control": len(negative_control),
    }


# Generate concentration-response plots
if not data_df.empty:
    print("📊 Generating concentration-response plots...")
    for feature in ["signal_mean", "signal_max", "signal_auc"]:
        if feature in data_df.columns:
            plot_concentration_response(data_df, feature)

    # Estimate detection limits
    detection_limits = estimate_detection_limits(data_df, "signal_mean")
    if detection_limits:
        print("\n📊 Detection Limits Estimation:")
        print("=" * 60)
        print("Baseline (negative control):")
        print(f"  Mean: {detection_limits['baseline_mean']:.2f}")
        print(f"  Std: {detection_limits['baseline_std']:.2f}")
        print(f"  N samples: {detection_limits['n_negative_control']}")
        print("\nLOD (Limit of Detection):")
        print(f"  Signal threshold: {detection_limits['lod_signal']:.2f}")
        if detection_limits["lod_concentration"] is not None:
            print(f"  Estimated concentration: {detection_limits['lod_concentration']:.2f} CFU/ml")
        print("\nLOQ (Limit of Quantification):")
        print(f"  Signal threshold: {detection_limits['loq_signal']:.2f}")
        if detection_limits["loq_concentration"] is not None:
            print(f"  Estimated concentration: {detection_limits['loq_concentration']:.2f} CFU/ml")
else:
    print("⚠️ No data available for concentration-response analysis")

In [ ]:
# Sensor Repeatability Analysis
# --------------------------------------------------------------------------------------------------


def calculate_sensor_repeatability(df: pd.DataFrame, feature: str = "signal_mean") -> pd.DataFrame:
    """Calculate repeatability metrics across sensors for each concentration.

    Returns DataFrame with:
    - concentration: Concentration level
    - n_sensors: Number of sensors tested
    - mean: Mean signal across sensors
    - std: Standard deviation across sensors
    - cv: Coefficient of variation (%)
    - min: Minimum signal
    - max: Maximum signal
    - range: Signal range (max - min)
    """
    if df.empty or feature not in df.columns:
        return pd.DataFrame()

    results = []

    for conc in sorted(df["concentration"].unique()):
        conc_data = df[df["concentration"] == conc]

        # Group by sensor and get mean per sensor
        sensor_means = conc_data.groupby("sensor_id")[feature].mean()

        if len(sensor_means) == 0:
            continue

        results.append(
            {
                "concentration": conc,
                "n_sensors": len(sensor_means),
                "mean": sensor_means.mean(),
                "std": sensor_means.std(),
                "cv": (sensor_means.std() / sensor_means.mean() * 100)
                if sensor_means.mean() > 0
                else np.nan,
                "min": sensor_means.min(),
                "max": sensor_means.max(),
                "range": sensor_means.max() - sensor_means.min(),
                "median": sensor_means.median(),
            }
        )

    return pd.DataFrame(results)


def plot_sensor_repeatability(repeatability_df: pd.DataFrame, feature: str = "signal_mean") -> None:
    """Plot sensor repeatability metrics."""
    if repeatability_df.empty:
        print("⚠️ No repeatability data to plot")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Experiment 1: Sensor Repeatability Analysis", fontsize=16, fontweight="bold")

    # Plot 1: CV vs Concentration
    ax1 = axes[0, 0]
    ax1.plot(
        repeatability_df["concentration"],
        repeatability_df["cv"],
        marker="o",
        linewidth=2,
        markersize=8,
        color="blue",
    )
    ax1.set_xlabel("Concentration (CFU/ml)")
    ax1.set_ylabel("Coefficient of Variation (%)")
    ax1.set_title("Sensor-to-Sensor Variability (CV) vs Concentration")
    ax1.set_xscale("log")
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=20, color="red", linestyle="--", label="20% CV threshold", alpha=0.7)
    ax1.legend()

    # Plot 2: Mean ± Std vs Concentration
    ax2 = axes[0, 1]
    ax2.errorbar(
        repeatability_df["concentration"],
        repeatability_df["mean"],
        yerr=repeatability_df["std"],
        marker="o",
        capsize=5,
        linewidth=2,
        markersize=8,
        color="green",
    )
    ax2.set_xlabel("Concentration (CFU/ml)")
    ax2.set_ylabel(feature.replace("_", " ").title())
    ax2.set_title("Mean Signal ± Std Across Sensors")
    ax2.set_xscale("log")
    ax2.grid(True, alpha=0.3)

    # Plot 3: Range vs Concentration
    ax3 = axes[1, 0]
    ax3.plot(
        repeatability_df["concentration"],
        repeatability_df["range"],
        marker="o",
        linewidth=2,
        markersize=8,
        color="orange",
    )
    ax3.set_xlabel("Concentration (CFU/ml)")
    ax3.set_ylabel("Signal Range (Max - Min)")
    ax3.set_title("Signal Range Across Sensors vs Concentration")
    ax3.set_xscale("log")
    ax3.grid(True, alpha=0.3)

    # Plot 4: Number of sensors vs Concentration
    ax4 = axes[1, 1]
    ax4.bar(range(len(repeatability_df)), repeatability_df["n_sensors"], color="purple", alpha=0.7)
    ax4.set_xticks(range(len(repeatability_df)))
    ax4.set_xticklabels([f"{c:.0f}" for c in repeatability_df["concentration"]], rotation=45)
    ax4.set_xlabel("Concentration (CFU/ml)")
    ax4.set_ylabel("Number of Sensors")
    ax4.set_title("Number of Sensors Tested per Concentration")
    ax4.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_sensor_boxplots(df: pd.DataFrame, feature: str = "signal_mean") -> None:
    """Plot boxplots showing sensor variability at each concentration."""
    if df.empty or feature not in df.columns:
        print("⚠️ No data available for boxplots")
        return

    # Select key concentrations for visualization
    concentrations = sorted(df["concentration"].unique())
    # Limit to reasonable number for visualization
    if len(concentrations) > 10:
        # Select evenly spaced concentrations
        indices = np.linspace(0, len(concentrations) - 1, 10, dtype=int)
        concentrations = [concentrations[i] for i in indices]

    fig, ax = plt.subplots(figsize=(14, 6))

    data_to_plot = []
    labels = []
    for conc in concentrations:
        conc_data = df[df["concentration"] == conc]
        if len(conc_data) > 0:
            # Get mean per sensor
            sensor_means = conc_data.groupby("sensor_id")[feature].mean().values
            data_to_plot.append(sensor_means)
            labels.append(f"{conc:.0f}")

    if data_to_plot:
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("lightblue")
            patch.set_alpha(0.7)
        ax.set_xlabel("Concentration (CFU/ml)")
        ax.set_ylabel(feature.replace("_", " ").title())
        ax.set_title("Sensor-to-Sensor Variability: Distribution at Each Concentration")
        ax.grid(True, axis="y", alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


# Calculate and plot sensor repeatability
if not data_df.empty:
    repeatability_df = calculate_sensor_repeatability(data_df, "signal_mean")

    if not repeatability_df.empty:
        print("✅ Sensor repeatability calculated")
        print("\n📊 Repeatability Summary:")
        print("=" * 60)
        display(repeatability_df)

        # Plot repeatability
        plot_sensor_repeatability(repeatability_df, "signal_mean")
        plot_sensor_boxplots(data_df, "signal_mean")

        # Summary statistics
        print("\n📊 Overall Repeatability Statistics:")
        print("=" * 60)
        print(f"Mean CV across all concentrations: {repeatability_df['cv'].mean():.2f}%")
        print(f"Median CV across all concentrations: {repeatability_df['cv'].median():.2f}%")
        print(
            f"CV Range: {repeatability_df['cv'].min():.2f}% - {repeatability_df['cv'].max():.2f}%"
        )

        # Protocol check: CV < 20% is generally considered good repeatability
        good_repeatability = repeatability_df[repeatability_df["cv"] < 20]
        print(
            f"\nConcentrations with good repeatability (CV < 20%): {len(good_repeatability)}/{len(repeatability_df)}"
        )
    else:
        print("⚠️ Could not calculate sensor repeatability")
else:
    print("⚠️ No data available for sensor repeatability analysis")

## 6. Serovar Comparison (Protocol Requirement)

Compare signal responses across the three serovars (Typhimurium, Hadar, Muenchen) to assess if sensors can differentiate between them.


In [ ]:
# Serovar Comparison Analysis
# --------------------------------------------------------------------------------------------------


def plot_serovar_comparison(df: pd.DataFrame, feature: str = "signal_mean") -> None:
    """Plot comparison of signal responses across serovars."""
    if df.empty or feature not in df.columns:
        print("⚠️ No data available for serovar comparison")
        return

    serotypes = sorted(df["serotype"].unique())
    concentrations = sorted(df["concentration"].unique())

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Experiment 1: Serovar Comparison", fontsize=16, fontweight="bold")

    # Plot 1: Concentration-response curves for each serovar
    ax1 = axes[0, 0]
    for serotype in serotypes:
        sero_data = df[df["serotype"] == serotype]
        concs = sorted(sero_data["concentration"].unique())
        means = [sero_data[sero_data["concentration"] == c][feature].mean() for c in concs]
        stds = [sero_data[sero_data["concentration"] == c][feature].std() for c in concs]
        ax1.errorbar(
            concs,
            means,
            yerr=stds,
            marker="o",
            label=serotype,
            capsize=5,
            linewidth=2,
            markersize=8,
        )
    ax1.set_xlabel("Concentration (CFU/ml)")
    ax1.set_ylabel(feature.replace("_", " ").title())
    ax1.set_title("Concentration-Response: Per-Serovar Comparison")
    ax1.set_xscale("log")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Boxplot comparison at each concentration
    ax2 = axes[0, 1]
    # Select key concentrations
    key_concs = concentrations[: min(6, len(concentrations))]  # Show up to 6 concentrations

    data_to_plot = []
    labels = []
    positions = []
    pos = 0

    for conc in key_concs:
        conc_data = df[df["concentration"] == conc]
        for serotype in serotypes:
            sero_conc_data = conc_data[conc_data["serotype"] == serotype]
            if len(sero_conc_data) > 0:
                data_to_plot.append(sero_conc_data[feature].values)
                labels.append(f"{serotype}\n{conc:.0f}")
                positions.append(pos)
                pos += 1

    if data_to_plot:
        bp = ax2.boxplot(data_to_plot, positions=positions, labels=labels, patch_artist=True)
        colors = plt.cm.Set3(np.linspace(0, 1, len(bp["boxes"])))
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax2.set_ylabel(feature.replace("_", " ").title())
        ax2.set_title("Signal Distribution: Serovar × Concentration")
        ax2.grid(True, axis="y", alpha=0.3)
        plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

    # Plot 3: Mean signal per serovar (bar chart)
    ax3 = axes[1, 0]
    sero_means = df.groupby("serotype")[feature].mean()
    bars = ax3.bar(range(len(sero_means)), sero_means.values, alpha=0.7)
    ax3.set_xticks(range(len(sero_means)))
    ax3.set_xticklabels(sero_means.index)
    ax3.set_ylabel(f"Mean {feature.replace('_', ' ').title()}")
    ax3.set_title("Overall Mean Signal per Serovar")
    ax3.grid(True, axis="y", alpha=0.3)

    # Add value labels on bars
    for bar, val in zip(bars, sero_means.values):
        height = bar.get_height()
        ax3.text(
            bar.get_x() + bar.get_width() / 2.0, height, f"{val:.1f}", ha="center", va="bottom"
        )

    # Plot 4: CV per serovar
    ax4 = axes[1, 1]
    sero_cvs = []
    sero_labels = []
    for serotype in serotypes:
        sero_data = df[df["serotype"] == serotype]
        if len(sero_data) > 0:
            mean_val = sero_data[feature].mean()
            std_val = sero_data[feature].std()
            cv = (std_val / mean_val * 100) if mean_val > 0 else np.nan
            sero_cvs.append(cv)
            sero_labels.append(serotype)

    if sero_cvs:
        bars = ax4.bar(sero_labels, sero_cvs, alpha=0.7, color="orange")
        ax4.set_ylabel("Coefficient of Variation (%)")
        ax4.set_title("Variability (CV) per Serovar")
        ax4.grid(True, axis="y", alpha=0.3)
        ax4.axhline(y=20, color="red", linestyle="--", label="20% CV threshold", alpha=0.7)
        ax4.legend()

        # Add value labels
        for bar, val in zip(bars, sero_cvs):
            height = bar.get_height()
            ax4.text(
                bar.get_x() + bar.get_width() / 2.0, height, f"{val:.1f}%", ha="center", va="bottom"
            )

    plt.tight_layout()
    plt.show()


# Generate serovar comparison plots
if not data_df.empty and len(data_df["serotype"].unique()) > 1:
    print("📊 Generating serovar comparison plots...")
    plot_serovar_comparison(data_df, "signal_mean")

    # Statistical comparison
    print("\n📊 Serovar Comparison Statistics:")
    print("=" * 60)
    for serotype in sorted(data_df["serotype"].unique()):
        sero_data = data_df[data_df["serotype"] == serotype]
        print(f"\n{serotype}:")
        print(f"  N samples: {len(sero_data)}")
        print(
            f"  Mean signal: {sero_data['signal_mean'].mean():.2f} ± {sero_data['signal_mean'].std():.2f}"
        )
        print(
            f"  Concentration range: {sero_data['concentration'].min():.1f} - {sero_data['concentration'].max():.1f} CFU/ml"
        )

    # Statistical test (if multiple serovars)
    if len(data_df["serotype"].unique()) >= 2:
        from scipy.stats import kruskal

        serotypes_list = sorted(data_df["serotype"].unique())
        groups = [data_df[data_df["serotype"] == s]["signal_mean"].values for s in serotypes_list]
        try:
            stat, p_value = kruskal(*groups)
            print("\n📊 Statistical Test (Kruskal-Wallis):")
            print(f"  H-statistic: {stat:.4f}")
            print(f"  p-value: {p_value:.4e}")
            if p_value < 0.05:
                print("  → Significant difference between serovars (p < 0.05)")
            else:
                print("  → No significant difference between serovars (p >= 0.05)")
        except Exception as e:
            print(f"  ⚠️ Could not perform statistical test: {e}")
elif len(data_df["serotype"].unique()) == 1:
    print("⚠️ Only one serotype available - cannot compare serovars")
    print(f"   Available serotype: {list(data_df['serotype'].unique())[0]}")
    print("   Protocol requires: ST (Typhimurium), Hadar, Muenchen")
else:
    print("⚠️ No data available for serovar comparison")

In [ ]:
# Summary Statistics and ML Training Data Preparation
# --------------------------------------------------------------------------------------------------


def generate_summary_statistics(df: pd.DataFrame) -> dict:
    """Generate comprehensive summary statistics for Experiment 1.

    Returns dictionary with:
    - per_concentration: Statistics grouped by concentration
    - per_serotype: Statistics grouped by serotype
    - per_sensor: Statistics grouped by sensor
    - overall: Overall statistics
    """
    if df.empty:
        return {}

    summaries = {}

    # Per-concentration summary
    conc_summary = (
        df.groupby("concentration")
        .agg(
            {
                "signal_mean": ["mean", "std", "min", "max", "count"],
                "signal_max": ["mean", "std"],
                "signal_auc": ["mean", "std"],
                "detection_status": "sum",
            }
        )
        .round(4)
    )
    summaries["per_concentration"] = conc_summary

    # Per-serotype summary
    if "serotype" in df.columns:
        sero_summary = (
            df.groupby("serotype")
            .agg(
                {
                    "signal_mean": ["mean", "std", "min", "max", "count"],
                    "concentration": ["min", "max", "mean"],
                    "detection_status": "sum",
                }
            )
            .round(4)
        )
        summaries["per_serotype"] = sero_summary

    # Per-sensor summary
    sensor_summary = (
        df.groupby("sensor_id")
        .agg(
            {
                "signal_mean": ["mean", "std", "min", "max", "count"],
                "concentration": ["min", "max", "nunique"],
                "detection_status": "sum",
            }
        )
        .round(4)
    )
    summaries["per_sensor"] = sensor_summary

    # Overall summary
    summaries["overall"] = {
        "total_samples": len(df),
        "n_sensors": df["sensor_id"].nunique(),
        "n_serotypes": df["serotype"].nunique() if "serotype" in df.columns else 0,
        "n_concentrations": df["concentration"].nunique(),
        "concentration_range": (df["concentration"].min(), df["concentration"].max()),
        "mean_signal": df["signal_mean"].mean(),
        "std_signal": df["signal_mean"].std(),
        "detection_rate": df["detection_status"].mean(),
    }

    return summaries


def prepare_ml_training_data(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare data in ML-ready format.

    Returns DataFrame with:
    - Features: signal features (mean, max, auc, etc.)
    - Labels: concentration (regression), detection_status (classification), serotype (classification)
    - Metadata: sensor_id, test_id, etc.
    """
    if df.empty:
        return pd.DataFrame()

    ml_df = df.copy()

    # Select features for ML
    feature_cols = [
        "signal_mean",
        "signal_max",
        "signal_min",
        "signal_std",
        "signal_range",
        "signal_auc",
        "signal_snr",
    ]

    # Keep only available features
    available_features = [col for col in feature_cols if col in ml_df.columns]

    # Labels
    label_cols = ["concentration", "log_concentration", "detection_status", "serotype"]
    available_labels = [col for col in label_cols if col in ml_df.columns]

    # Metadata
    metadata_cols = ["sensor_id", "test_id", "filename", "signal_index"]
    available_metadata = [col for col in metadata_cols if col in ml_df.columns]

    # Select columns
    ml_columns = available_features + available_labels + available_metadata
    ml_df = ml_df[ml_columns].copy()

    return ml_df


# Generate summaries
if not data_df.empty:
    summaries = generate_summary_statistics(data_df)

    print("📊 Summary Statistics:")
    print("=" * 80)

    if "overall" in summaries:
        print("\nOverall Summary:")
        for key, value in summaries["overall"].items():
            print(f"  {key}: {value}")

    if "per_concentration" in summaries:
        print("\n📊 Per-Concentration Summary:")
        print("=" * 80)
        display(summaries["per_concentration"])

    if "per_serotype" in summaries:
        print("\n📊 Per-Serotype Summary:")
        print("=" * 80)
        display(summaries["per_serotype"])

    if "per_sensor" in summaries:
        print("\n📊 Per-Sensor Summary (first 10 sensors):")
        print("=" * 80)
        display(summaries["per_sensor"].head(10))

    # Prepare ML training data
    ml_training_data = prepare_ml_training_data(data_df)
    if not ml_training_data.empty:
        print("\n✅ ML Training Data Prepared:")
        print(f"   Shape: {ml_training_data.shape}")
        print(
            f"   Features: {[col for col in ml_training_data.columns if col.startswith('signal_')]}"
        )
        print(
            f"   Labels: {[col for col in ml_training_data.columns if col in ['concentration', 'detection_status', 'serotype']]}"
        )
        print("\n   Sample data:")
        display(ml_training_data.head())
    else:
        print("⚠️ Could not prepare ML training data")
else:
    print("⚠️ No data available for summary statistics")